# Verify Hantaviridae PCR-positive host species

This notebook reproduces the filtering logic used to get host species with **positive PCR detections** for viruses in **Hantaviridae** and provides an explicit check for `Mustela nivalis`.


In [ ]:
import pandas as pd
from pathlib import Path

CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in CANDIDATES if (path / "data" / "processed").exists())
ARHA_DIR = REPO_ROOT / "data" / "processed" / "arha"
SDM_DIR = REPO_ROOT / "data" / "processed" / "sdm"
host_path = ARHA_DIR / "host_minimal.csv"
pcr_path = ARHA_DIR / "pathogen_taxonomy_minimal.csv"
out_csv = SDM_DIR / "hantaviridae_pcr_serology_positive_host_species.csv"
out_txt = SDM_DIR / "hantaviridae_pcr_serology_positive_host_species.txt"

host = pd.read_csv(host_path)
pcr = pd.read_csv(pcr_path)

print('host rows:', len(host))
print('pcr rows:', len(pcr))
print('host columns:', list(host.columns))
print('pcr columns:', list(pcr.columns))
print('assay types:', pcr['assay'].unique())

pcr

In [ ]:
# Core filter logic
assay = pcr['assay'].astype(str).str.strip().str.lower()
family = pcr['assigned_family'].fillna(pcr['pathogen_family']).astype(str).str.strip().str.lower()
number_positive = pd.to_numeric(pcr['number_positive'], errors='coerce').fillna(0)

mask = family.eq('hantaviridae') & (number_positive > 0)
pcr_pos = pcr.loc[mask].copy()

print('Rows passing filter (PCR + Hantaviridae + number_positive>0):', len(pcr_pos))


In [ ]:
# Join to host species and summarize
host_lookup = host[['host_record_id', 'host_species']].copy()
merged = pcr_pos.merge(host_lookup, on='host_record_id', how='left')
merged['host_species'] = merged['host_species'].astype(str).str.strip()
merged = merged[merged['host_species'].str.lower() != 'nan'].copy()

species_summary = (
    merged.groupby('host_species', dropna=False)
    .agg(
        positive_pcr_rows=('host_record_id', 'size'),
        total_number_positive=('number_positive', 'sum'),
        unique_host_record_ids=('host_record_id', 'nunique'),
    )
    .reset_index()
    .sort_values('host_species')
)

print('Unique host species with positive PCR Hantaviridae detections:', species_summary['host_species'].nunique())
species_summary.head(20)


In [ ]:
# Save outputs (same format used in your workflow)
species_summary.to_csv(out_csv, index=False)
species_summary[['host_species']].to_csv(out_txt, index=False, header=False)

print('Saved:', out_csv)
print('Saved:', out_txt)


In [ ]:
# Explicit audit for Mustela nivalis
target = 'mustela nivalis'

host_target = host[host['host_species'].astype(str).str.strip().str.lower().eq(target)].copy()
target_ids = set(host_target['host_record_id'].astype(str))

print('Mustela nivalis rows in host table:', len(host_target))
print('Unique host_record_id for Mustela nivalis:', len(target_ids))
print('host_record_id list:', sorted(target_ids))

pcr_target = pcr[pcr['host_record_id'].astype(str).isin(target_ids)].copy()
pcr_target['family_effective'] = pcr_target['assigned_family'].fillna(pcr_target['pathogen_family'])
pcr_target['number_positive_num'] = pd.to_numeric(pcr_target['number_positive'], errors='coerce').fillna(0)

hanta_target = pcr_target[pcr_target['family_effective'].astype(str).str.strip().str.lower().eq('hantaviridae')].copy()
hanta_target_pos = hanta_target[hanta_target['number_positive_num'] > 0].copy()

print('PCR rows for Mustela nivalis host_record_ids:', len(pcr_target))
print('Hantaviridae rows among those:', len(hanta_target))
print('Positive Hantaviridae rows among those:', len(hanta_target_pos))
print('Sum(number_positive) in Hantaviridae rows:', hanta_target['number_positive_num'].sum())

hanta_target[[
    'pathogen_record_id', 'host_record_id', 'assay', 'pathogen_species_cleaned',
    'family_effective', 'number_tested', 'number_positive'
]].sort_values(['host_record_id', 'pathogen_record_id'])


In [ ]:
# Optional: compare with previously saved CSV if it exists
if out_csv.exists():
    saved = pd.read_csv(out_csv)
    print('Saved CSV rows:', len(saved))
    print('Current summary rows:', len(species_summary))
    print('Row count match:', len(saved) == len(species_summary))
else:
    print('No saved CSV found yet at', out_csv)
